# rCNV BURDEN TEST

In [ ]:
import pandas as pd
import os
import sys
import subprocess

In [ ]:
my_bucket = os.getenv('WORKSPACE_BUCKET')

# Modify Covariate Files: Adding Pheno and Burden

In [ ]:
df_cov_all = pd.read_csv('all_covariates_ALL_AUD_age_sex_pcs_SVPrunedRelatedness.txt',
                         sep='\t') # This is covariates file
df_aud_all = pd.read_csv('phenotypes.tsv',
                        sep='\t') # This file contains IID and Phenotype columns

In [ ]:
df_cov_all_pheno = pd.merge(df_cov_all, df_aud_all[['IID', 'Phenotype']], on='IID', how='left')
# df_cov_all_pheno.rename(columns={'Age': 'AGE'}, inplace=True)
df_cov_all_pheno.to_csv('all_covariates_ALL_AUD_age_sex_pcs_SVPrunedRelatedness_pheno.txt',
                         sep='\t', index=False)

In [ ]:
import pandas as pd
import os
import glob

input_dir = "Cases_Controls_Within_Ancestry/"
output_dir = input_dir 

# Pattern: only files that contain SVPrunedRelatedness but NOT KeepList
pattern = os.path.join(input_dir, 
                       "all_covariates_WITHIN_*_AUD_age_sex_pcs_SVPrunedRelatedness.txt")

files = glob.glob(pattern)

print(f"Found {len(files)} files")

for file in files:
    
    print(f"Processing: {os.path.basename(file)}")
    
    df_cov = pd.read_csv(file, sep="\t")
    
    df_cov_pheno = pd.merge(
        df_cov,
        df_aud_all[['IID', 'Phenotype']],
        on='IID',
        how='left'
    )
    
    new_filename = file.replace(".txt", "_pheno.txt")
    
    df_cov_pheno.to_csv(new_filename, sep="\t", index=False)

print("Done.")


In [ ]:
burden_df = pd.read_csv('gene_set_analysis/results/TOTAL_COMBINED_BURDEN_MASTER_all_CNV_sizes.tsv',
                       sep='\t')

In [ ]:
burden_df_all = burden_df[burden_df['gene_set'] == 'Burden_Summary_all_rare_cnvs_to_extract']
col_to_add = ['IID', 'genome_wide_total_burden_mb', 'genome_wide_total_count', 'average_cnv_length_mb']
df_cov_final = pd.merge(df_cov_all_pheno, burden_df_all[col_to_add], on='IID', how='left')

df_cov_final = df_cov_final.rename(columns={'Age' : 'AGE',
                                            'genome_wide_total_burden_mb': 'TOTAL_BURDEN',
                                           'genome_wide_total_count' : 'TOTAL_COUNT',
                                           'average_cnv_length_mb' : 'AVERAGE_LENGTH'})

df_cov_final.to_csv('aud/AUD_Cases_Controls_EHR_Depth_Ancestry_PoPMaD/all_covariates_ALL_AUD_age_sex_pcs_SVPrunedRelatedness_pheno_CountLength.txt',
                   sep='\t', index=False)

# GLM Logistic Regression

# Firth's Logistic Regression

In [ ]:
%%writefile gene_set_analysis/scripts/analyze_cnv_firth_burden_association.R
#!/usr/bin/env Rscript

# R Script: analyze_cnv_firth_burden_association.R
# Purpose: Performs Firth penalized logistic regression with detailed logging.

suppressPackageStartupMessages({
    library(data.table)
    library(logistf)
})

args <- commandArgs(trailingOnly = TRUE)
if (length(args) < 3) {
    stop("Usage: Rscript analyze_cnv_firth_burden_association.R <burden_file> <phenotype_covariate_file> <output_results_file> <output_log_file>")
}

burden_file              <- args[1]
phenotype_covariate_file <- args[2]
output_results_file      <- args[3]

# Create log filename if not provided as 4th argument
output_log_file <- if(length(args) == 4) args[4] else gsub("\\.[^.]*$", ".log", output_results_file)

# --- 1. INITIALIZE LOG ---
log_con <- file(output_log_file, open = "wt")
cat(paste("Firth Regression Run Date:", Sys.time(), "\n"), file = log_con)
cat(paste("Burden File:", burden_file, "\n"), file = log_con)
cat(paste("Phenotype File:", phenotype_covariate_file, "\n\n"), file = log_con)

# --- 2. LOAD DATA ---
dt_burden <- fread(burden_file)
dt_pheno  <- fread(phenotype_covariate_file)

# Standardize IDs
dt_burden[, IID := as.character(tolower(trimws(IID)))]
dt_pheno[,  IID := as.character(tolower(trimws(IID)))]

# Remove FID or other overlaps from pheno to prevent .x/.y suffixes
common_cols <- intersect(names(dt_burden), names(dt_pheno))
common_cols <- common_cols[common_cols != "IID"]
if(length(common_cols) > 0) dt_pheno[, (common_cols) := NULL]

dt_merged <- merge(dt_pheno, dt_burden, by = "IID")
cat(paste("Merged N:", nrow(dt_merged), "individuals\n"), file = log_con)

if (nrow(dt_merged) == 0) {
    cat("FATAL: Zero individuals merged. Check ID formatting.\n", file = log_con)
    close(log_con)
    stop("FATAL: Zero individuals merged.")
}

# --- 3. SET PHENOTYPE ---
all_cols <- names(dt_merged)
PHENO_COL <- all_cols[trimws(all_cols) == "AUD"][1]
if (is.na(PHENO_COL)) {
    PHENO_COL <- all_cols[grep("AUD", trimws(all_cols), ignore.case = TRUE)][1]
}

if (is.na(PHENO_COL)) {
    cat("FATAL: Phenotype column 'AUD' not found.\n", file = log_con)
    close(log_con)
    stop("FATAL: Phenotype column not found.")
}
cat(paste("Selected Phenotype Column:", PHENO_COL, "\n"), file = log_con)

# --- 4. DEFINE COVARIATES ---
### The script automatically identifies the variable names from the covariate table and import
### into the model. SO FOR EACH MODEL, COMMENT OUT AND RUN EACH AT A TIME.

# --------- MODEL I --------- ##
# potential_covariates <- c("AGE", "SEX", "PC1", "PC2", "PC3", "PC4", "PC5", "PC6", 
#                           "PC7", "PC8", "PC9", "PC10")

## --------- MODEL II --------- ## BEST MODEL
# potential_covariates <- c("AGE", "SEX", "PC1", "PC2", "PC3", "PC4", "PC5", "PC6", 
#                           "PC7", "PC8", "PC9", "PC10", "TOTAL_COUNT", "AVERAGE_LENGTH")

## --------- MODEL III --------- ##
potential_covariates <- c("AGE", "SEX", "PC1", "PC2", "PC3", "PC4", "PC5", "PC6", 
                          "PC7", "PC8", "PC9", "PC10", "TOTAL_BURDEN", "AVERAGE_LENGTH")

## --------- MODEL IV --------- ##
# potential_covariates <- c("AGE", "SEX", "PC1", "PC2", "PC3", "PC4", "PC5", "PC6", 
#                           "PC7", "PC8", "PC9", "PC10", "TOTAL_BURDEN", "AVERAGE_LENGTH", "TOTAL_COUNT")


covariates_used <- potential_covariates[potential_covariates %in% names(dt_merged)]
covar_part <- paste(covariates_used, collapse = " + ")

cat("Covariates detected and used in Firth model:\n", file = log_con)
if(length(covariates_used) > 0) {
    cat(paste("-", covariates_used, collapse = "\n"), file = log_con, append = TRUE)
} else {
    cat("- NONE (Running unadjusted)", file = log_con, append = TRUE)
}
cat("\n\n", file = log_con)

# --- 5. RUN MODELS ---
burden_vars <- c("genome_wide_del_burden_mb", "genome_wide_dup_burden_mb", 
                 "genome_wide_total_burden_mb", "genome_wide_del_count", "genome_wide_dup_count")

results_list <- list()

for (var in burden_vars) {
    if (!var %in% names(dt_merged)) {
        cat(paste("Skipping", var, ": Missing from data\n"), file = log_con)
        next
    }
    
    vals <- na.omit(dt_merged[[var]])
    if (uniqueN(vals) < 2) {
        cat(paste("Skipping", var, ": No variation (all values identical)\n"), file = log_con)
        next
    }

    # Build formula with backticks for safety
    form_str <- if(length(covariates_used) > 0) {
        paste0("`", PHENO_COL, "` ~ `", var, "` + ", covar_part)
    } else {
        paste0("`", PHENO_COL, "` ~ `", var, "`")
    }
    
    tryCatch({
        # Firth Regression (flic = TRUE for intercept correction)
        model <- logistf(as.formula(form_str), data = dt_merged, pl = FALSE, flic = TRUE)
        
        target <- paste0("`", var, "`")
        idx <- which(names(model$coefficients) == target | names(model$coefficients) == var)

        if (length(idx) == 1) {
            beta <- model$coefficients[idx]
            se   <- sqrt(diag(model$var))[idx]
            
            results_list[[var]] <- data.table(
                Burden_Metric = var,
                N_Individuals = nrow(dt_merged),
                Beta          = beta,
                Std_Error     = se,
                Odds_Ratio    = exp(beta),
                OR_L95        = exp(beta - 1.96 * se),
                OR_U95        = exp(beta + 1.96 * se),
                P_Value       = model$prob[idx]
            )
        }
    }, error = function(e) {
        cat(paste("ERROR in", var, ":", e$message, "\n"), file = log_con)
    })
}

# --- 6. SAVE AND CLOSE ---
final_results <- rbindlist(results_list)
if (nrow(final_results) > 0) {
    fwrite(final_results, output_results_file, sep = "\t")
    cat(paste("Successfully saved results to:", output_results_file, "\n"), file = log_con)
} else {
    cat("No results were generated for any burden metric.\n", file = log_con)
}

close(log_con)
message(paste("Process complete. Log available at:", output_log_file))

# Shell Script to run the R script for each phenotype file

In [ ]:
%%writefile gene_set_analysis/scripts/cnv_analysis_step03.sh
#!/bin/bash
set -euo pipefail

# --- 1. DEFINE PATHS (Input Localization Fix) ---
# Use the explicit DSUB_LOCAL_INPUT_ variables for local paths.
LOCAL_R_SCRIPT="${r_script_analysis}"
LOCAL_BURDEN_FILE="${burden_file}"
LOCAL_PHENO_FILE="${pheno_cov_file}"

# Define a simple, local temporary path for the R script to write to.
TEMP_FINAL_RESULTS_FILE="/mnt/data/cnv_association_results.tsv"
TEMP_LOG_FILE="/mnt/data/cnv_association_results.log"

# The complex path where dsub expects the final file to be found is ${out_file}.
# We will copy to this path explicitly.

echo "--- 1. STARTING CNV BURDEN ASSOCIATION ANALYSIS ---"
echo "Burden File (Local VM Path): ${LOCAL_BURDEN_FILE}"
echo "Phenotype File (Local VM Path): ${LOCAL_PHENO_FILE}"
echo "R Script Temp Output Path: ${TEMP_FINAL_RESULTS_FILE}"

# --- 2. CRITICAL CHECK: Input File Localization ---
if [ ! -f "${LOCAL_BURDEN_FILE}" ]; then
    echo "FATAL: Burden file not found at ${LOCAL_BURDEN_FILE}. Check GCS access."
    exit 1
fi

# --- 3. RUN R SCRIPT ---
# ARGUMENTS: <R_script> <burden_file> <phenotype_covariate_file> <output_results_file>
# The R script will write results to the TEMP file path.
Rscript "${LOCAL_R_SCRIPT}" \
    "${LOCAL_BURDEN_FILE}" \
    "${LOCAL_PHENO_FILE}" \
    "${TEMP_FINAL_RESULTS_FILE}" \
    "${TEMP_LOG_FILE}"

# --- 4. OUTPUT HANDLING (DELOCALIZATION FIX using the CP pattern) ---

if [ -f "${TEMP_FINAL_RESULTS_FILE}" ]; then
    
    # CRITICAL FIX: The target destination for dsub's delocalization is ${out_file}.
    # We must explicitly copy the result from the simple TEMP path to the complex 
    # path represented by the shell variable ${out_file}.
    echo "Copying final results from ${TEMP_FINAL_RESULTS_FILE} -> ${out_file}"
    
    # Ensure the destination directory exists before copying.
    TARGET_DIR=$(dirname "${out_file}")
    mkdir -p "${TARGET_DIR}"

    cp "${TEMP_FINAL_RESULTS_FILE}" "${out_file}"
    
    echo "Copying log file..."
    cp "${TEMP_LOG_FILE}" "${out_log}"
    
    echo "SUCCESS: Association analysis complete and results copied for delocalization."
else
    echo "FATAL: Analysis R script failed to produce the output file: ${TEMP_FINAL_RESULTS_FILE}"
    exit 1
fi

echo "--- ANALYSIS JOB COMPLETE ---"

In [ ]:
%%bash

source ~/aou_dsub.bash

# Define the two burden categories
# CATEGORIES=("all" "10kb")
CATEGORIES=("10kb")

METHODS=("glm" "firth")

## --- INPUT FILES HAVE DIFFERENT FILE NAME PATTERNS SO RUN EACH WITH CORRESPONDING BURDEN_FILE_GCS BELOW

NAMES=$(gsutil ls ${WORKSPACE_BUCKET}/data/gene_sets_ndd/results/POPMaD_ALL/burden_scores_summary/*.tsv | \
  sed -n 's/.*_in_set_\(.*\)\.tsv/\1/p' | tr '\n' ' ')
GENE_SETS=($NAMES)


total_tasks=$(( ${#CATEGORIES[@]} * ${#GENE_SETS[@]} * ${#METHODS[@]} ))
current_task=0


for size in "${CATEGORIES[@]}"; do
    for GSS in "${GENE_SETS[@]}"; do
        for METHOD in "${METHODS[@]}"; do

            # 1. Map to correct R script
            if [ "$METHOD" == "firth" ]; then
                R_SCRIPT_NAME="analyze_cnv_firth_burden_association.R"
            else
                R_SCRIPT_NAME="analyze_cnv_glm_burden_association.R"
            fi

            # 2. Define Paths
            R_SCRIPT_GCS="${WORKSPACE_BUCKET}/data/gene_sets/scripts/${R_SCRIPT_NAME}"
            

            BURDEN_FILE_GCS="${WORKSPACE_BUCKET}/data/gene_sets_ndd/results/POPMaD_ALL/burden_scores_summary/POPMaD_ALL_Burden_Summary_ndd_${size}_cnv_ids_present_in_set_${GSS}.tsv"


            PHENO_COV_GCS="${WORKSPACE_BUCKET}/data/cnv_phenotypes_covariates/all_covariates_ALL_AUD_age_sex_pcs_SVPrunedRelatedness_pheno_CountLength.txt"
            SHELL_SCRIPT_ANALYSIS="${WORKSPACE_BUCKET}/data/gene_sets/scripts/cnv_analysis_step03.sh"

            # 3. Dynamic Output Path (organized by method and then category)
#             FINAL_RESULTS_FILE="${WORKSPACE_BUCKET}/data/gene_sets_ndd/results/POPMaD_ALL/burden_association/${METHOD}/genomewide_${size}_${GSS}_association_10PCsAgeSex.tsv"
#             FINAL_RESULTS_FILE="${WORKSPACE_BUCKET}/data/gene_sets_ndd/results/POPMaD_ALL/burden_association/${METHOD}/genomewide_${size}_${GSS}_association_10PCsAgeSexTotalCountAvgLength.tsv"
            FINAL_RESULTS_FILE="${WORKSPACE_BUCKET}/data/gene_sets_ndd/results/POPMaD_ALL/burden_association/${METHOD}/genomewide_${size}_${GSS}_association_10PCsAgeSexTotalLengthAvgLength.tsv"
#             FINAL_RESULTS_FILE="${WORKSPACE_BUCKET}/data/gene_sets_ndd/results/POPMaD_ALL/burden_association/${METHOD}/genomewide_${size}_${GSS}_association_10PCsAgeSexTotalLengthAvgLengthTotalCount.tsv"
            
            FINAL_LOG_FILE="${FINAL_RESULTS_FILE%.tsv}.log"
            
            job_name="${GSS}_${METHOD}"
            job_name=$(echo "${job_name}" | cut -c 1-60 | sed 's/-*$//')
            
            ((current_task++))
            tasks_left=$((total_tasks - current_task))
      
            printf '=%.0s' {1..50}; echo ""
            echo "${BURDEN_FILE_GCS}"
            printf '=%.0s' {1..60}; echo ""
            echo "Task: [${current_task}/${total_tasks}] | Left: ${tasks_left}"
            echo "Submitting: GENE SET: ${GSS} | Size: ${size} | Method: ${METHOD}"
            printf '=%.0s' {1..60}; echo ""

            aou_dsub \
              --image us-central1-docker.pkg.dev/polar-standard-455018-c9/prscs-repo/prscs-suite-v1.0:1.1 \
              --name "${job_name}" \
              --disk-size 200 \
              --machine-type "n2-standard-4" \
              --input pheno_cov_file="${PHENO_COV_GCS}" \
              --input burden_file="${BURDEN_FILE_GCS}" \
              --input r_script_analysis="${R_SCRIPT_GCS}" \
              --output out_file="${FINAL_RESULTS_FILE}" \
              --output out_log="${FINAL_LOG_FILE}" \
              --script "${SHELL_SCRIPT_ANALYSIS}"
        done
    done
done

# Merge Results

In [ ]:
import pandas as pd
import os
import re

# Define the directory path
directory = "gene_set_analysis/NDD_gene_sets/results/burden_association/glm"

# List to store individual dataframes
dfs = []

# Regex pattern to capture:
# 1. size: (10kb|all)
# 2. gene_set: the text between 'AUD_' or 'rare_' and '_association'
# Note: This pattern is flexible for your specific naming convention
pattern = re.compile(r"genomewide_(?P<size>10kb|all)_(?P<gene_set>.+)_association_")

for filename in os.listdir(directory):
    if filename.endswith(".tsv"):
        match = pattern.search(filename)
        if match:
            # Extract metadata from filename
            size = match.group('size')
            gene_set = match.group('gene_set')
            
            # Load the TSV
            file_path = os.path.join(directory, filename)
            temp_df = pd.read_csv(file_path, sep='\t', dtype=str)
            
            # Add metadata columns
            temp_df['Size'] = size
            temp_df['Gene_Set'] = gene_set
            
            # Optional: Extract the covariate model from the end of the filename 
            # (e.g., 10PCsAgeSexTotalCountAvgLength)
            covariates = filename.split('_association_')[1].replace('.tsv', '')
            temp_df['Covariates'] = covariates
            
            dfs.append(temp_df)

# Merge all into one dataframe
final_df = pd.concat(dfs, ignore_index=True)

# Reorder columns to put metadata at the front
cols = ['Size', 'Gene_Set', 'Covariates'] + [c for c in final_df.columns if c not in ['Size', 'Gene_Set', 'Covariates']]
final_df = final_df[cols]

# Save the combined result
final_df.to_csv("gene_set_analysis/NDD_gene_sets/results/burden_association/glm/glm_merged_burden_results.tsv", 
                sep='\t', index=False, float_format=None)

print(f"Successfully merged {len(dfs)} files into 'glm_merged_burden_results.tsv'")
# print(final_df.head())

In [ ]:
import pandas as pd
import os
import re

# Define the directory path
directory = "gene_set_analysis/NDD_gene_sets/results/burden_association/firth"

# List to store individual dataframes
dfs = []

# Regex pattern to capture:
# 1. size: (10kb|all)
# 2. gene_set: the text between 'AUD_' or 'rare_' and '_association'
# Note: This pattern is flexible for your specific naming convention
pattern = re.compile(r"genomewide_(?P<size>10kb|all)_(?P<gene_set>.+)_association_")

for filename in os.listdir(directory):
    if filename.endswith(".tsv"):
        match = pattern.search(filename)
        if match:
            # Extract metadata from filename
            size = match.group('size')
            gene_set = match.group('gene_set')
            
            # Load the TSV
            file_path = os.path.join(directory, filename)
            temp_df = pd.read_csv(file_path, sep='\t', dtype=str)
            
            # Add metadata columns
            temp_df['Size'] = size
            temp_df['Gene_Set'] = gene_set
            
            # Optional: Extract the covariate model from the end of the filename 
            # (e.g., 10PCsAgeSexTotalCountAvgLength)
            covariates = filename.split('_association_')[1].replace('.tsv', '')
            temp_df['Covariates'] = covariates
            
            dfs.append(temp_df)

# Merge all into one dataframe
final_df = pd.concat(dfs, ignore_index=True)

# Reorder columns to put metadata at the front
cols = ['Size', 'Gene_Set', 'Covariates'] + [c for c in final_df.columns if c not in ['Size', 'Gene_Set', 'Covariates']]
final_df = final_df[cols]

# Save the combined result
final_df.to_csv("gene_set_analysis/NDD_gene_sets/results/burden_association/firth/firth_merged_burden_results.tsv", 
                sep='\t', index=False, float_format=None)

print(f"Successfully merged {len(dfs)} files into 'firth_merged_burden_results.tsv'")
# print(final_df.head())

# Statistical Analysis

In [ ]:
# Covariates: Age Sex 10PCs Total Count Avg Length

glm_df = pd.read_csv("gene_set_analysis/NDD_gene_sets/results/burden_association/glm/glm_merged_burden_results.tsv", 
                sep='\t', dtype=str)
firth_df = pd.read_csv("gene_set_analysis/NDD_gene_sets/results/burden_association/firth/firth_merged_burden_results.tsv", 
                sep='\t', dtype=str)
glm_df   = glm_df[glm_df['Covariates'] == '10PCsAgeSexTotalCountAvgLength'].copy()
firth_df = firth_df[firth_df['Covariates'] == '10PCsAgeSexTotalCountAvgLength'].copy()

In [ ]:
cols_to_fix = ['Beta', 'Std_Error', 'Odds_Ratio', 'P_Value']
glm_df[cols_to_fix] = glm_df[cols_to_fix].astype(float)
firth_df[cols_to_fix] = firth_df[cols_to_fix].astype(float)

In [ ]:
import numpy as np

# -log10P and Z-score calculations

glm_df['P_Value'] = glm_df['P_Value'].astype(np.float64)
glm_df['-log10_P'] = -np.log10(glm_df['P_Value'])

firth_df['P_Value'] = firth_df['P_Value'].astype(np.float64)
firth_df['-log10_P'] = -np.log10(firth_df['P_Value'])

glm_df['Z'] = glm_df['Beta'] / glm_df['Std_Error']
firth_df['Z'] = firth_df['Beta'] / firth_df['Std_Error']

In [ ]:
# FDR and Bonferroni

def apply_corrections(group):
    # Bonferroni Correction
    # returns: (reject_mask, pvals_corrected, alphac_sidak, alphac_bonf)
    group['P_Bonferroni'] = multipletests(group['P_Value'], method='bonferroni')[1]
    
    # FDR (Benjamini-Hochberg)
    group['P_FDR'] = multipletests(group['P_Value'], method='fdr_bh')[1]
    
    return group

# Apply the function to each group
glm_df_corrected   = glm_df.groupby(['Gene_Set'], group_keys=False).apply(apply_corrections)
firth_df_corrected = firth_df.groupby(['Gene_Set'], group_keys=False).apply(apply_corrections)

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(glm_df['Z'], firth_df['Z'], s=4, alpha=0.7, color='black')
plt.xlabel('GLM Z-score')
plt.ylabel('Firth Z-score')
plt.title('Z Statistics')
plt.axline((0, 0), slope=1, color='red', linestyle='--')
plt.show()

In [ ]:
# # 1. Make sure the correction is actually saved to a variable
# glm_df_corrected = glm_df.groupby(['Size', 'Gene_Set', 'Covariates'], group_keys=False).apply(apply_corrections)

# 2. Use that new variable for the plot
plt.figure(figsize=(10, 6), dpi=300)

# Plot all points using glm_df_corrected
plt.scatter(glm_df_corrected['Beta'], glm_df_corrected['-log10_P'], s=5, color='grey', alpha=0.5)

# Now 'P_FDR' will exist in this specific dataframe
significant = glm_df_corrected[glm_df_corrected['P_FDR'] < 0.05]
plt.scatter(significant['Beta'], significant['-log10_P'], s=5, color='red', label='FDR < 0.05')

plt.axhline(-np.log10(0.05), color='blue', linestyle='--', alpha=0.5, label='Nominal p = 0.05')
plt.xlabel('Beta')
plt.ylabel('-log10(P)')
plt.title('Logistic Regression\nBurden Test Results (Covariates: Age Sex 10PCs Total Count and Average Length)')
plt.legend()
plt.show()

In [ ]:
# # 1. Make sure the correction is actually saved to a variable
# glm_df_corrected = glm_df.groupby(['Size', 'Gene_Set', 'Covariates'], group_keys=False).apply(apply_corrections)

# 2. Use that new variable for the plot
plt.figure(figsize=(10, 6), dpi=300)

# Plot all points using glm_df_corrected
plt.scatter(firth_df_corrected['Beta'], firth_df_corrected['-log10_P'], s=5, color='grey', alpha=0.5)

# Now 'P_FDR' will exist in this specific dataframe
significant = firth_df_corrected[firth_df_corrected['P_FDR'] < 0.05]
plt.scatter(significant['Beta'], significant['-log10_P'], s=5, color='red', label='FDR < 0.05')

plt.axhline(-np.log10(0.05), color='blue', linestyle='--', alpha=0.5, label='Nominal p = 0.05')
plt.xlabel('Beta')
plt.ylabel('-log10(P)')
plt.title("Firth's Regression\nBurden Test Results (Covariates: Age Sex 10PCs Total Count and Average Length)")
plt.legend()